# Model Gate v2 — 6-class, ArcFace, 384px

The upgraded model gate. Same discipline as v1 (baselines → real approaches → pick on
**validation** → sealed test **once** → reloadable artifacts), with three research-backed
changes aimed at the two things v1 exposed — the **Cobalt/Gentra/Nexia 3 look-alikes** and
the need to reject **unknown** cars:

1. **6 classes** — adds the open-set `others` class (derived automatically from the folders).
2. **Sub-center ArcFace head (K=3)** — an angular-margin head that widens the tiny gap
   between the look-alike sedans and tolerates weak labels. *Zero* inference cost (the
   margin is train-time only). This is the headline upgrade.
3. **384px fine-tune** — the distinguishing cues (grille slats, light internals, badge) are
   sub-pixel at 224; training at 384 is the cheapest reliable gain.

Augmentation is **label-preserving and fine-grained-safe** (RandomResizedCrop with a high
scale floor, flip, RandAugment, Random Erasing). We deliberately **do not** use vanilla
MixUp/CutMix — on look-alike cars they erase the one discriminative patch. An optional
**SnapMix** arm (semantic-proportion labels) is provided for the FG-aware mixing experiment.

**Before running:** the **6-class** `dataset_split.zip` must be in Drive `CapstoneCars/`,
and set **Runtime → T4 GPU**. 384px is ~3× the compute of 224 — a full run of the two main
approaches is ≈ 60–90 min; run the model cells one at a time.

> The test set is untouched until **Cell 11**. We pick the winner on `val/` only.

In [ ]:
# ── Cell 1 · Setup ──────────────────────────────────────────────
!pip install -q timm mlflow
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, random, json, glob
from pathlib import Path
from torch.amp import autocast, GradScaler
import timm
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = torch.cuda.is_available()
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", dev, "| timm", timm.__version__)

In [ ]:
# ── Cell 2 · Load the 6-class leakage-safe split from Drive ─────
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/dataset
zips = glob.glob('/content/drive/MyDrive/**/dataset_split.zip', recursive=True)
print("found:", zips)
ZIP = zips[0]
!unzip -o -q "$ZIP" -d /content
DATA = Path('/content/dataset')
for split in ['train','val','test']:
    print(split, {d.name: len(list(d.glob('*'))) for d in sorted((DATA/split).iterdir())})

In [ ]:
# ── Cell 3 · Dataloaders @384 + FG-safe aug + class weights ─────
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG   = 384                                        # high-res: sub-pixel cues become visible
BS    = 24                                         # T4-safe at 384; drop to 16 if you hit OOM
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)

# Fine-grained-safe, label-PRESERVING augmentation (no MixUp/CutMix).
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.65,1.0)),   # high scale floor keeps the car whole
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=7),        # mild; heavy aug hurts fine-grained
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),                      # occlusion robustness
])
eval_tf = transforms.Compose([
    transforms.Resize(438), transforms.CenterCrop(IMG),    # 438 ≈ 384 / 0.875
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_ds = datasets.ImageFolder(DATA/'train', train_tf)
val_ds   = datasets.ImageFolder(DATA/'val',   eval_tf)
test_ds  = datasets.ImageFolder(DATA/'test',  eval_tf)
train_loader = DataLoader(train_ds, BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BS, shuffle=False, num_workers=2, pin_memory=True)

CLASSES = train_ds.classes                 # alphabetical: cobalt, damas, gentra, nexia3, others, spark
class_to_idx = train_ds.class_to_idx
counts  = np.bincount([y for _,y in train_ds.samples], minlength=len(CLASSES))
weights = torch.tensor(counts.sum()/(len(CLASSES)*counts), dtype=torch.float32).to(dev)
assert len(CLASSES) == 6, f"expected 6 classes, got {CLASSES}"
print("classes:", CLASSES)
print("train counts:", dict(zip(CLASSES, counts.tolist())))
print("class weights:", weights.cpu().numpy().round(3))

In [ ]:
# ── Cell 4 · MLflow + shared train/eval helpers ─────────────────
import os, mlflow
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'   # 2026 MLflow: opt into file-based store
mlflow.set_tracking_uri('file:/content/mlruns')
mlflow.set_experiment('uzbek-car-modelgate-v2')
results, models = {}, {}

P1_EPOCHS, P2_EPOCHS = 2, 6     # phase 1 = head warmup (frozen backbone), phase 2 = unfreeze

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for x,y in loader:
        with autocast('cuda', enabled=USE_AMP):
            out = model(x.to(dev))                 # eval path: no labels -> margin-free logits
        ps.append(out.argmax(1).cpu()); ys.append(y)
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy()
    return {'acc': float(accuracy_score(y,p)),
            'macro_f1': float(f1_score(y,p,average='macro'))}, y, p

def train_model(model, epochs, lr_groups, tag, arcface=False, patience=3):
    model.to(dev)
    opt = torch.optim.AdamW(lr_groups, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss(weight=weights)
    scaler = GradScaler('cuda', enabled=USE_AMP)
    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train(); tot = 0.0
        for x,y in train_loader:
            x,y = x.to(dev), y.to(dev)
            opt.zero_grad()
            with autocast('cuda', enabled=USE_AMP):
                out = model(x, y) if arcface else model(x)   # ArcFace needs labels for the margin
                loss = lossf(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()*x.size(0)
        sched.step()
        m,_,_ = evaluate(model, val_loader)
        print(f"[{tag}] ep {ep+1}/{epochs}  loss {tot/len(train_ds):.3f}  "
              f"val_acc {m['acc']:.3f}  val_macroF1 {m['macro_f1']:.3f}")
        if m['macro_f1'] > best_f1:
            best_f1 = m['macro_f1']
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state)
    return model, best_f1

def log_run(name, params, metrics):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.log_metrics({'val_'+k: v for k,v in metrics.items()})

In [ ]:
# ── Cell 5 · Baseline · Majority class (the 6-class floor) ──────
from collections import Counter
maj  = Counter([y for _,y in train_ds.samples]).most_common(1)[0][0]
yval = np.array([y for _,y in val_ds.samples])
results['majority'] = {'acc': float((yval==maj).mean()),
                       'macro_f1': float(f1_score(yval, np.full_like(yval, maj), average='macro'))}
log_run('baseline-majority', {'strategy': f'always {CLASSES[maj]}'}, results['majority'])
print(f"majority ({CLASSES[maj]}):", results['majority'])

In [ ]:
# ── Cell 6 · Model definitions (ArcFace + SnapMix heads) ────────
# NOTE: the Trust Layer notebook must include these SAME class definitions to reload the
# model if the ArcFace/SnapMix winner is chosen (they are custom modules, not plain timm).

class SubCenterArcFace(nn.Module):
    """Angular-margin head with K sub-centers per class (Deng 2019 / Deng 2020).
    Training: adds margin m to the target class angle. Inference (labels=None): plain s*cos."""
    def __init__(self, in_features, num_classes, K=3, s=30.0, m=0.30):
        super().__init__()
        self.num_classes, self.K, self.s, self.m = num_classes, K, s, m
        self.W = nn.Parameter(torch.empty(num_classes*K, in_features))
        nn.init.xavier_uniform_(self.W)
    def forward(self, feat, labels=None):
        f = F.normalize(feat, dim=1)
        w = F.normalize(self.W, dim=1)
        cos = (f @ w.t()).view(-1, self.num_classes, self.K).amax(dim=2)   # sub-center max
        cos = cos.float().clamp(-1+1e-6, 1-1e-6)                           # fp32 for stable acos
        if labels is None:
            return self.s * cos
        theta  = torch.acos(cos)
        margin = torch.zeros_like(cos).scatter_(1, labels.view(-1,1), self.m)
        return self.s * torch.cos(theta + margin)

class ArcModel(nn.Module):
    """timm backbone (global-avg-pooled features) + sub-center ArcFace head."""
    def __init__(self, model_id, num_classes, K=3, s=30.0, m=0.30, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_id, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        self.head = SubCenterArcFace(self.backbone.num_features, num_classes, K, s, m)
    def forward(self, x, labels=None):
        return self.head(self.backbone(x), labels)
    def head_params(self):     return list(self.head.parameters())
    def backbone_params(self): return list(self.backbone.parameters())

class SnapMixNet(nn.Module):
    """Backbone feature map + own GAP + linear head, so CAM (for SnapMix) is exact."""
    def __init__(self, model_id, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_id, pretrained=pretrained,
                                          num_classes=0, global_pool='')
        self.C  = self.backbone.num_features
        self.fc = nn.Linear(self.C, num_classes)
    def feat_map(self, x):  return self.backbone.forward_features(x)   # (B,C,H,W)
    def forward(self, x):
        f = self.feat_map(x)
        return self.fc(F.adaptive_avg_pool2d(f, 1).flatten(1))
    def spm(self, x, labels):                                          # semantic percent map
        f = self.feat_map(x)
        w = self.fc.weight[labels]                                     # (B,C)
        return F.relu(torch.einsum('bchw,bc->bhw', f, w))
    def head_params(self):     return list(self.fc.parameters())
    def backbone_params(self): return list(self.backbone.parameters())

print("model classes defined: SubCenterArcFace, ArcModel, SnapMixNet")

In [ ]:
# ── Cell 7 · Approach A · ConvNeXt-Tiny plain softmax @384 ──────
# Resolution-upgraded baseline: isolates what 384px alone buys over v1's 224px.
MODEL_ID = 'convnext_tiny.fb_in22k_ft_in1k'
plain = timm.create_model(MODEL_ID, pretrained=True, num_classes=len(CLASSES))
p_head = list(plain.get_classifier().parameters())

for p in plain.parameters(): p.requires_grad = False   # phase 1: head only
for p in p_head:             p.requires_grad = True
plain, _ = train_model(plain, P1_EPOCHS, [{'params':p_head,'lr':1e-3}], 'plain384-p1')

for p in plain.parameters(): p.requires_grad = True    # phase 2: unfreeze, discriminative LRs
p_head_ids = {id(p) for p in p_head}
p_back = [p for p in plain.parameters() if id(p) not in p_head_ids]
plain, _ = train_model(plain, P2_EPOCHS,
                       [{'params':p_back,'lr':3e-5},{'params':p_head,'lr':3e-4}], 'plain384-p2')

results['plain_384'], models['plain_384'] = evaluate(plain, val_loader)[0], plain
log_run('convnext-plain-384', {'arch':MODEL_ID,'img':IMG,'p1':P1_EPOCHS,'p2':P2_EPOCHS},
        results['plain_384'])
print("plain @384 val:", results['plain_384'])

In [ ]:
# ── Cell 8 · Approach B (PRIMARY) · Sub-center ArcFace @384 ─────
# Same backbone, angular-margin head — targets the Cobalt/Gentra/Nexia3 look-alikes.
ARC = {'K':3, 's':30.0, 'm':0.30}
arc = ArcModel(MODEL_ID, len(CLASSES), pretrained=True, **ARC)

for p in arc.backbone.parameters(): p.requires_grad = False   # phase 1: head only
arc, _ = train_model(arc, P1_EPOCHS, [{'params':arc.head_params(),'lr':1e-2}],
                     'arc384-p1', arcface=True)

for p in arc.backbone.parameters(): p.requires_grad = True    # phase 2: unfreeze
arc, _ = train_model(arc, P2_EPOCHS,
                     [{'params':arc.backbone_params(),'lr':3e-5},
                      {'params':arc.head_params(),'lr':3e-3}], 'arc384-p2', arcface=True)

results['arcface_384'], models['arcface_384'] = evaluate(arc, val_loader)[0], arc
log_run('convnext-arcface-384', {'arch':MODEL_ID,'img':IMG,'head':'subcenter_arcface',**ARC},
        results['arcface_384'])
print("ArcFace @384 val:", results['arcface_384'])

In [ ]:
# ── Cell 9 · (OPTIONAL) Approach C · SnapMix @384 ──────────────
# FG-aware mixing: replace a box, but weight the two labels by the SEMANTIC mass (CAM)
# moved — so the label matches what is actually visible. Uses single-box geometry (robust)
# with SnapMix-style semantic-proportion labels. Set RUN_SNAPMIX = True to train it.
RUN_SNAPMIX = False

def rand_bbox(H, W, lam):
    cut = float(np.sqrt(1.0 - lam)); ch, cw = int(H*cut), int(W*cut)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1,y2 = np.clip(cy-ch//2,0,H), np.clip(cy+ch//2,0,H)
    x1,x2 = np.clip(cx-cw//2,0,W), np.clip(cx+cw//2,0,W)
    return y1,y2,x1,x2

def train_snapmix(model, epochs, lr_groups, tag, beta=5.0, patience=3):
    model.to(dev)
    opt = torch.optim.AdamW(lr_groups, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    cef = nn.CrossEntropyLoss(weight=weights, reduction='none')
    scaler = GradScaler('cuda', enabled=USE_AMP)
    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train(); tot = 0.0
        for x,y in train_loader:
            x,y = x.to(dev), y.to(dev); B,_,H,W = x.shape
            with torch.no_grad():                              # per-image semantic percent map
                spm = model.spm(x, y)
                spm = spm / (spm.flatten(1).sum(1).view(B,1,1) + 1e-8)
            idx = torch.randperm(B, device=dev)
            lam = float(np.random.beta(beta, beta))
            y1,y2,x1,x2 = rand_bbox(H, W, lam)                 # box replaced in target by source
            x_mix = x.clone(); x_mix[:,:,y1:y2,x1:x2] = x[idx][:,:,y1:y2,x1:x2]
            gh, gw = spm.shape[1], spm.shape[2]                # box on the CAM grid
            sy1,sy2 = int(y1/H*gh), max(int(y2/H*gh), int(y1/H*gh)+1)
            sx1,sx2 = int(x1/W*gw), max(int(x2/W*gw), int(x1/W*gw)+1)
            prop_t = spm[:,     sy1:sy2, sx1:sx2].flatten(1).sum(1)   # mass removed from target
            prop_s = spm[idx][:, sy1:sy2, sx1:sx2].flatten(1).sum(1)  # mass added from source
            la, lb = (1.0 - prop_t), prop_s
            opt.zero_grad()
            with autocast('cuda', enabled=USE_AMP):
                out = model(x_mix)
                loss = (cef(out, y)*la + cef(out, y[idx])*lb).mean()
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()*B
        sched.step()
        m,_,_ = evaluate(model, val_loader)
        print(f"[{tag}] ep {ep+1}/{epochs}  loss {tot/len(train_ds):.3f}  "
              f"val_acc {m['acc']:.3f}  val_macroF1 {m['macro_f1']:.3f}")
        if m['macro_f1'] > best_f1:
            best_f1 = m['macro_f1']
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state); return model, best_f1

if RUN_SNAPMIX:
    snap = SnapMixNet(MODEL_ID, len(CLASSES), pretrained=True)
    for p in snap.backbone.parameters(): p.requires_grad = False
    snap, _ = train_snapmix(snap, P1_EPOCHS, [{'params':snap.head_params(),'lr':1e-3}], 'snap384-p1')
    for p in snap.backbone.parameters(): p.requires_grad = True
    snap, _ = train_snapmix(snap, P2_EPOCHS,
                            [{'params':snap.backbone_params(),'lr':3e-5},
                             {'params':snap.head_params(),'lr':3e-4}], 'snap384-p2')
    results['snapmix_384'], models['snapmix_384'] = evaluate(snap, val_loader)[0], snap
    log_run('convnext-snapmix-384', {'arch':MODEL_ID,'img':IMG,'aug':'snapmix'}, results['snapmix_384'])
    print("SnapMix @384 val:", results['snapmix_384'])
else:
    print("SnapMix arm skipped (set RUN_SNAPMIX=True to train it).")

In [ ]:
# ── Cell 10 · Compare on VALIDATION and pick the winner ─────────
import pandas as pd
tbl = pd.DataFrame(results).T[['acc','macro_f1']].sort_values('macro_f1', ascending=False)
print("VALIDATION comparison (sorted by macro-F1):")
print(tbl.round(4))
winner = next(n for n in tbl.index if n in models)     # best TRAINED model
print("\nSelected model:", winner)

In [ ]:
# ── Cell 11 · SEALED test (winner only, once) + error analysis ──
best_model = models[winner]
tm, ytrue, ypred = evaluate(best_model, test_loader)
print(f"=== SEALED TEST — {winner} ===")
print(f"test_acc {tm['acc']:.3f}   test_macroF1 {tm['macro_f1']:.3f}")
print(classification_report(ytrue, ypred, target_names=CLASSES, digits=3))

# precision on the 5 KNOWN models (the business signal; 'others' is the reject bucket)
oth = class_to_idx['others']
known = ytrue != oth
known_acc = float((ypred[known]==ytrue[known]).mean())
print(f"accuracy on the 5 known classes (excluding 'others'): {known_acc:.3f}")

import matplotlib.pyplot as plt
cm = confusion_matrix(ytrue, ypred)
fig, ax = plt.subplots(figsize=(5.5,4.5)); ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=8)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(f'{winner} — test confusion (watch the sedan trio + others)')
plt.tight_layout(); plt.savefig('/content/confusion_v2.png', dpi=120); plt.show()

with mlflow.start_run(run_name=f'{winner}-TEST'):
    mlflow.log_metrics({'test_acc':tm['acc'],'test_macro_f1':tm['macro_f1'],'known5_acc':known_acc})
    mlflow.log_artifact('/content/confusion_v2.png')

In [ ]:
# ── Cell 12 · Save reloadable artifacts + clean reload check ────
ART = Path('/content/artifacts'); ART.mkdir(exist_ok=True)
torch.save(best_model.state_dict(), ART/'model.pt')
HEAD_OF = {'plain_384':'linear', 'arcface_384':'arcface', 'snapmix_384':'snapmix'}
cfg = {'winner':winner, 'head':HEAD_OF[winner], 'model_id':MODEL_ID, 'img_size':IMG,
       'mean':MEAN, 'std':STD, 'class_to_idx':class_to_idx, 'classes':CLASSES}
if HEAD_OF[winner] == 'arcface': cfg['arc'] = ARC
json.dump(cfg, open(ART/'config.json','w'), indent=2)

def build_model(cfg):
    h = cfg['head']
    if   h == 'arcface': m = ArcModel(cfg['model_id'], len(cfg['classes']), pretrained=False, **cfg['arc'])
    elif h == 'snapmix': m = SnapMixNet(cfg['model_id'], len(cfg['classes']), pretrained=False)
    else:                m = timm.create_model(cfg['model_id'], pretrained=False, num_classes=len(cfg['classes']))
    m.load_state_dict(torch.load(ART/'model.pt', map_location=dev))
    return m.to(dev).eval()

from PIL import Image
reloaded = build_model(json.load(open(ART/'config.json')))
path, true = test_ds.samples[0]
x = eval_tf(Image.open(path).convert('RGB')).unsqueeze(0).to(dev)
with torch.no_grad(), autocast('cuda', enabled=USE_AMP):
    pred = reloaded(x).argmax(1).item()
print("reload OK — head:", cfg['head'], "| predicted:", CLASSES[pred], "| true:", CLASSES[true])
print("artifacts:", [p.name for p in ART.iterdir()])

In [ ]:
# ── Cell 13 · Back up artifacts + MLflow runs to Drive ──────────
!cd /content && zip -q -r modelgate_v2_artifacts.zip artifacts mlruns confusion_v2.png >/dev/null
!cp /content/modelgate_v2_artifacts.zip /content/drive/MyDrive/CapstoneCars/
print("✅ v2 model + config + MLflow runs backed up to Drive as modelgate_v2_artifacts.zip")
print("   config:", {k: cfg[k] for k in ['winner','head','model_id','img_size']})